## pengujian model kelulusan smk yang tersimpan dengan form inputan slidebar

In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

# Memastikan model kelulusan SMK sudah dimuat
# Jika belum, pastikan sel 80a5d1f5 sudah dijalankan untuk menyimpan model
# dan dbbde962 untuk memuatnya sebagai 'loaded_model'
if 'loaded_model' not in globals():
    try:
        loaded_model = tf.keras.models.load_model('model_kelulusan_smk.keras')
        print("Model 'model_kelulusan_smk.keras' berhasil dimuat kembali.")
    except Exception as e:
        print(f"Error memuat model kelulusan SMK: {e}. Pastikan model sudah disimpan sebagai 'model_kelulusan_smk.keras'.")
        loaded_model = None

# Memastikan scaler_large sudah dimuat dari sel 29aa284c
if 'scaler_large' not in globals():
    print("Membuat scaler_large dummy karena tidak ditemukan. Harap pastikan sel '29aa284c' dijalankan untuk scaler yang benar.")
    scaler_large = StandardScaler()
    # Fit scaler dengan data contoh jika tidak ada
    # Ini sangat penting agar scaler memiliki statistik (mean, std) untuk transformasi
    try:
        # Asumsi X_large ada dari 29aa284c
        _X_dummy_large = np.random.uniform(low=[0, 60, 30], high=[25, 100, 100], size=(10, 3))
        scaler_large.fit(_X_dummy_large)
    except NameError:
        print("Tidak dapat memfit scaler_large dummy, variabel 'X_large' dari sel sebelumnya tidak ditemukan.")


# Membuat widget input (slider)
jam_belajar_slider = widgets.IntSlider(
    value=10,
    min=0,
    max=25,
    step=1,
    description='Jam Belajar:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

kehadiran_slider = widgets.IntSlider(
    value=80,
    min=20,
    max=100,
    step=1,
    description='% Kehadiran:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

nilai_praktik_slider = widgets.IntSlider(
    value=70,
    min=30,
    max=100,
    step=1,
    description='Nilai Praktik:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

predict_button_smk = widgets.Button(
    description='Prediksi Kelulusan',
    disabled=False,
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Klik untuk mendapatkan prediksi kelulusan'
)

output_area_smk = widgets.Output()

def on_predict_button_smk_clicked(b):
    with output_area_smk:
        clear_output()
        if loaded_model is None:
            print("Model belum dimuat. Tidak dapat melakukan prediksi.")
            return

        # Ambil nilai dari slider
        jam_belajar = jam_belajar_slider.value
        kehadiran = kehadiran_slider.value
        nilai_praktik = nilai_praktik_slider.value

        # Buat array numpy dari input
        student_data = np.array([[jam_belajar, kehadiran, nilai_praktik]])

        # Skalakan data input
        if 'scaler_large' in globals():
            student_data_scaled = scaler_large.transform(student_data)
        else:
            print("Scaler `scaler_large` tidak ditemukan, tidak dapat menskalakan data. Prediksi mungkin tidak akurat.")
            student_data_scaled = student_data # Lanjutkan tanpa scaling jika scaler tidak ditemukan (hati-hati)

        # Lakukan prediksi
        prediction_prob = loaded_model.predict(student_data_scaled, verbose=0)[0][0]
        prediction_status = 'LULUS' if prediction_prob >= 0.5 else 'TIDAK LULUS'

        # Tampilkan hasil
        print(f"\nData Siswa: [Jam Belajar: {jam_belajar}, % Kehadiran: {kehadiran}, Nilai Praktik: {nilai_praktik}]")
        print(f"Probabilitas Kelulusan: {prediction_prob * 100:.2f}%")
        print(f"Status Prediksi: {prediction_status}")

# Hubungkan tombol dengan fungsi
predict_button_smk.on_click(on_predict_button_smk_clicked)

print("Form Interaktif Prediksi Kelulusan SMK")
# Tampilkan widget
display(jam_belajar_slider, kehadiran_slider, nilai_praktik_slider, predict_button_smk, output_area_smk)


I0000 00:00:1783925297.597308   21505 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1783925297.688969   21505 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783925301.794331   21505 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Model 'model_kelulusan_smk.keras' berhasil dimuat kembali.
Membuat scaler_large dummy karena tidak ditemukan. Harap pastikan sel '29aa284c' dijalankan untuk scaler yang benar.
Form Interaktif Prediksi Kelulusan SMK


E0000 00:00:1783925304.284137   21505 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


IntSlider(value=10, continuous_update=False, description='Jam Belajar:', max=25)

IntSlider(value=80, continuous_update=False, description='% Kehadiran:', min=20)

IntSlider(value=70, continuous_update=False, description='Nilai Praktik:', min=30)

Button(button_style='info', description='Prediksi Kelulusan', style=ButtonStyle(), tooltip='Klik untuk mendapa…

Output()

## Pengujian model dengan upload file csv

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from IPython.display import display, clear_output
import io
import ipywidgets as widgets

# Memastikan model kelulusan SMK sudah dimuat
if 'loaded_model' not in globals():
    try:
        loaded_model = tf.keras.models.load_model('model_kelulusan_smk.keras')
        print("Model 'model_kelulusan_smk.keras' berhasil dimuat kembali.")
    except Exception as e:
        print(f"Error memuat model kelulusan SMK: {e}. Pastikan model sudah disimpan sebagai 'model_kelulusan_smk.keras'.")
        loaded_model = None

# Memastikan scaler_large sudah dimuat dari sel sebelumnya
if 'scaler_large' not in globals():
    print("Membuat scaler_large dummy karena tidak ditemukan. Harap pastikan sel sebelumnya dijalankan untuk scaler yang benar.")
    scaler_large = StandardScaler()
    try:
        _X_dummy_large = np.random.uniform(low=[0, 60, 30], high=[25, 100, 100], size=(10, 3))
        scaler_large.fit(_X_dummy_large)
    except Exception as e:
        print(f"Tidak dapat memfit scaler_large dummy: {e}")


upload_widget = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Pilih File CSV'
)

predict_button = widgets.Button(
    description='Prediksi CSV',
    button_style='info'
)

output_area = widgets.Output()


def on_predict_button_clicked(_):
    with output_area:
        clear_output(wait=True)

        if loaded_model is None:
            print("Model kelulusan SMK belum dimuat. Tidak dapat melakukan prediksi dari CSV.")
            return

        if 'scaler_large' not in globals():
            print("Scaler `scaler_large` tidak tersedia. Tidak dapat melakukan prediksi yang akurat dari CSV.")
            return

        if not upload_widget.value:
            print("Belum ada file CSV yang dipilih.")
            return

        uploaded_value = upload_widget.value

        if isinstance(uploaded_value, dict):
            uploaded_file = next(iter(uploaded_value.values()))
            file_name = uploaded_file.get('name', 'file.csv')
            file_content = uploaded_file.get('content')
        elif isinstance(uploaded_value, (tuple, list)):
            if len(uploaded_value) == 0:
                print("Belum ada file CSV yang dipilih.")
                return
            first_item = uploaded_value[0]
            if isinstance(first_item, tuple) and len(first_item) >= 2:
                file_name = first_item[0]
                file_content = first_item[1]
            elif isinstance(first_item, dict):
                file_name = first_item.get('name', 'file.csv')
                file_content = first_item.get('content')
            else:
                print("Format file upload tidak didukung.")
                return
        else:
            print("Format file upload tidak didukung.")
            return

        if file_content is None:
            print("File tidak dapat dibaca.")
            return

        print(f"Memproses file: {file_name}")
        try:
            if isinstance(file_content, str):
                df_input = pd.read_csv(io.StringIO(file_content))
            else:
                df_input = pd.read_csv(io.BytesIO(file_content))
        except Exception as e:
            print(f"Gagal membaca file CSV: {e}")
            return

        expected_columns = ['Jam Belajar', 'Kehadiran', 'Nilai Praktik']
        if not all(col in df_input.columns for col in expected_columns):
            print("Error: File CSV harus memiliki kolom 'Jam Belajar', 'Kehadiran', dan 'Nilai Praktik'.")
            return

        student_data_from_csv = df_input[expected_columns].to_numpy()
        student_data_scaled_from_csv = scaler_large.transform(student_data_from_csv)

        predictions_probabilities_csv = loaded_model.predict(student_data_scaled_from_csv, verbose=0)
        predictions_classes_csv = (predictions_probabilities_csv >= 0.5).astype(int)

        df_input['Probabilitas Kelulusan'] = [f'{p[0]*100:.2f}%' for p in predictions_probabilities_csv]
        df_input['Status Prediksi'] = ['LULUS' if c[0] == 1 else 'TIDAK LULUS' for c in predictions_classes_csv]

        print("\nHasil Prediksi Kelulusan dari File CSV:")
        display(df_input)


predict_button.on_click(on_predict_button_clicked)
display(upload_widget, predict_button, output_area)

FileUpload(value=(), accept='.csv', description='Pilih File CSV')

Button(button_style='info', description='Prediksi CSV', style=ButtonStyle())

Output()